# 03 Multi-Model Training & Calibration Pipeline
**Smart India Hackathon 2026 — Problem Statement 26102**

This notebook demonstrates the training, loss convergence, and serialization of the three AI/ML sub-models:
1. **Statistical Anomaly Detector**:
   - *Isolation Forest*: Unsupervised spatial partitioning across expenditure-progress contours.
   - *Deep Autoencoder (PyTorch)*: Non-linear reconstruction error quantifying multivariate divergence.
2. **Irregularity & Fraud Classifier (XGBoost)**:
   - Stratified supervised classifier mapping complex interactions to financial irregularity propensity.
3. **Efficiency & Timeline Regressor (Gradient Boosting Regressor)**:
   - Estimates empirical project duration without target leakage to benchmark execution pacing.

In [ ]:
import os
import sys
import pickle
from pathlib import Path
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))
from src.config import MODELS_DIR

# Check trained model artifacts
models = {
    "Isolation Forest": MODELS_DIR / "isolation_forest.pkl",
    "PyTorch Autoencoder": MODELS_DIR / "autoencoder.pt",
    "XGBoost Fraud Classifier": MODELS_DIR / "fraud_classifier.pkl",
    "Efficiency Regressor": MODELS_DIR / "efficiency_regressor.pkl",
    "Ensemble Config": MODELS_DIR / "ensemble_config.pkl"
}

print("Model Artifact Inventory:")
for name, path in models.items():
    exists = path.exists()
    size_kb = path.stat().st_size / 1024 if exists else 0
    print(f" - {name:<30}: {'EXISTS' if exists else 'MISSING'} ({size_kb:.1f} KB)")

## 1. Inspecting XGBoost Fraud Classifier Artifact
Evaluating precision, recall, F1, and ROC-AUC on hold-out validation set.

In [ ]:
fraud_path = MODELS_DIR / "fraud_classifier.pkl"
if fraud_path.exists():
    with open(fraud_path, "rb") as f:
        fraud_payload = pickle.load(f)
    
    metrics = fraud_payload.get('metrics', {})
    print("Fraud Classifier Validation Performance:")
    for k, v in metrics.items():
        print(f"  {k.upper():<12}: {v:.4f}")
        
    # Top feature importances
    top_feats = sorted(fraud_payload.get('feature_importances', {}).items(), key=lambda x: x[1], reverse=True)[:10]
    f_names = [x[0] for x in top_feats]
    f_scores = [x[1] for x in top_feats]
    
    plt.figure(figsize=(10, 4))
    plt.barh(f_names[::-1], f_scores[::-1], color="#EF4444")
    plt.title("Top 10 Feature Importances (XGBoost Fraud Classifier)")
    plt.xlabel("Relative Importance Score")
    plt.show()

## 2. Inspecting Efficiency Regressor Artifact
Evaluating target-leakage-free execution duration prediction.

In [ ]:
eff_path = MODELS_DIR / "efficiency_regressor.pkl"
if eff_path.exists():
    with open(eff_path, "rb") as f:
        eff_payload = pickle.load(f)
    
    eff_metrics = eff_payload.get('metrics', {})
    print("Efficiency Regressor Evaluation Performance:")
    for k, v in eff_metrics.items():
        print(f"  {k.upper():<12}: {v:.4f}")